In [ ]:
# Import the libraries

%pip install yfinance 
%pip install plotly
%pip install scikit-learn

import yfinance as yf

import pandas as pd 
import numpy as np 

import plotly.graph_objects as go 
import plotly.express as px 
#import plotly.io as pio


In [ ]:
# Download the stock data from yfinance
import yfinance as yf
# Define the stock ticker and the date range
tickers = ['AAPL']
#tickers.append('GC=F')  # Gold futures
#tickers.append('BTC-USD')  # Bitcoin in USD
start_date = '2020-01-01'
end_date = '2026-01-01'

# Download the stock data
stocks = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True)
gold = yf.download('GC=F', start=start_date, end=end_date, auto_adjust=True)
BTC = yf.download('BTC-USD', start=start_date, end=end_date, auto_adjust=True)

# Arrange the data so that 'Close' price can directly be used for plotting and calculations
stocks.columns = stocks.columns.droplevel(1)  # Drop the ticker level from columns
gold.columns = gold.columns.droplevel(1)
BTC.columns = BTC.columns.droplevel(1)

# Merge the normalized closing price of gold and bitcoin with the stock data
#stocks['Gold'] = (gold['Close'] - np.mean(gold['Close'])) / np.std(gold['Close'])
#stocks['Bitcoin'] = (BTC['Close'] - np.mean(BTC['Close'])) / np.std(BTC['Close'])

# reset the index to make 'Date' a column
#stocks.reset_index(inplace=True)
# Drop the date column
#stocks.drop(columns=['Date'], inplace=True)

stocks.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800
2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200
2020-01-07,71.928040,72.533080,71.708680,72.277563,108872000
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200


In [ ]:
# Normalize the gold and bitcoin prices to the first value to compare relative changes
def normalize_to_first(df):
    return df / df.iloc[0]

stocks['Gold'] = normalize_to_first(gold['Close'])
stocks['Bitcoin'] = normalize_to_first(BTC['Close'])
#stocks[['Gold', 'Bitcoin']] = stocks[['Gold', 'Bitcoin']].apply(normalize_to_first)

stocks.head(20)

Price,Close,High,Low,Open,Volume,Gold,Bitcoin
Date,,,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400,1.000000,0.970181
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800,1.016202,1.020098
2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200,1.027353,1.079032
2020-01-07,71.928040,72.533080,71.708680,72.277563,108872000,1.031027,1.133819
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.021581,1.122176
2020-01-09,74.637474,74.830314,73.810661,74.061352,170108400,1.017842,1.094289
2020-01-10,74.806244,75.370316,74.304855,74.871333,140644800,1.021646,1.134216
2020-01-13,76.404427,76.430946,75.003904,75.122026,121532000,1.015677,1.131111
2020-01-14,75.372711,76.551468,75.249779,76.341752,161954400,1.011742,1.226049


In [ ]:
gold.head()

Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2020-01-02,1524.500000,1528.699951,1518.000000,1518.099976,214
2020-01-03,1549.199951,1552.699951,1530.099976,1530.099976,107
2020-01-06,1566.199951,1580.000000,1560.400024,1580.000000,416
2020-01-07,1571.800049,1576.300049,1558.300049,1558.300049,47
2020-01-08,1557.400024,1604.199951,1552.300049,1579.699951,236


In [ ]:
### Add Bollinger Bands to the stock data
# Calculate the 20-day moving average and standard deviation


def Bollinger_Bands(df, window=20, num_std_dev=2):
    df['MA20'] = df['Close'].rolling(window=window).mean()
    df['STD20'] = df['Close'].rolling(window=window).std()
    df['UpperBand'] = df['MA20'] + (df['STD20'] * num_std_dev)
    df['LowerBand'] = df['MA20'] - (df['STD20'] * num_std_dev)
    return df
stocks = Bollinger_Bands(stocks)  
#stocks.dropna(inplace=True)
#.reset_index(inplace=True)  # Drop rows with NaN values resulting from rolling calculations

stocks.head()

# Plot the stock price and Bollinger Bands
#pio.renderers.default = "browser"
def plot_bollinger_bands(stocks):
    fig = go.Figure().update_layout(width=1000, height=600, title='AAPL Stock Price with Bollinger Bands', xaxis_title='Date', yaxis_title='Price (USD)')
    fig.add_trace(go.Scatter(x=stocks.index, y=stocks['Close'], mode='lines', name='Close Price'))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['UpperBand'], 
                            mode = 'lines', 
                            name='Upper Bollinger Band', 
                            line=dict(color='red', dash='dot')))

    fig.add_trace(go.Scatter(x=stocks.index, 
                            y=stocks['LowerBand'], 
                            mode = 'lines', 
                            name='Lower Bollinger Band', 
                            line=dict(color='red', dash='dot')))
    fig.show()


In [ ]:
# Get 20 day, 50 day, and 200 day moving averages
stocks['MA50'] = stocks['Close'].rolling(window=50).mean()
stocks['MA200'] = stocks['Close'].rolling(window=200).mean()

# Get the lag features for the closing price
stocks['Lag1'] = stocks['Close'].shift(1)
stocks['Lag2'] = stocks['Close'].shift(2)

# Capture volume spikes
stocks['VolumeSpike'] = stocks['Volume'] > (stocks['Volume'].rolling(window=20).mean() + 2*stocks['Volume'].rolling(window=20).std())

# 
def RSI(df, period=14):
    delta = df['Close'].diff()
    gains = delta.where(delta > 0, 0)
    losses = -delta.where(delta < 0, 0)
    avg_gains = gains.rolling(window=period).mean()
    avg_losses = losses.rolling(window=period).mean()
    rs = avg_gains/avg_losses
    #print(rs)
    df['RSI'] = 100 - (100/(1 + rs))
    return df

RSI(stocks, 14)

# MACD - Moving Average Convergence Divergence
def MACD(df, short_window=12, long_window=26, signal_window=9):
    df['EMA12'] = df['Close'].ewm(span=short_window).mean()
    df['EMA26'] = df['Close'].ewm(span=long_window).mean()
    df['MACD'] = df['EMA12'] - df['EMA26']
    df['SignalLine'] = df['MACD'].ewm(span=signal_window).mean()
    df['MACD_Histogram'] = df['MACD'] - df['SignalLine']
    return df

MACD(stocks)
###stocks.dropna(inplace=True)
#stocks.head(20)


Price,Close,High,Low,Open,Volume,Gold,Bitcoin,MA20,STD20,UpperBand,...,VolumeSpike,RSI,EMA12,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400,1.000000,0.970181,NaN,NaN,NaN,...,False,NaN,72.400513,72.400513,0.000000,0.000000,0.000000,3,1,1
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800,1.016202,1.020098,NaN,NaN,NaN,...,False,NaN,72.019252,72.035044,-0.015792,-0.008773,-0.007019,4,1,1
2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200,1.027353,1.079032,NaN,NaN,NaN,...,False,NaN,72.116308,72.118715,-0.002407,-0.006164,0.003757,0,1,1
2020-01-07,71.928040,72.533080,71.708680,72.277563,108872000,1.031027,1.133819,NaN,NaN,NaN,...,False,NaN,72.056879,72.065410,-0.008531,-0.006966,-0.001565,1,1,1
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.021581,1.122176,NaN,NaN,NaN,...,False,NaN,72.336249,72.301885,0.034363,0.005329,0.029035,2,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-24,273.554016,275.172497,271.945536,272.085389,17910600,2.939062,12.168033,276.892905,4.242292,285.377488,...,False,33.540427,273.983287,273.680820,0.302467,1.443821,-1.141354,2,12,4
2025-12-26,273.144409,275.112569,272.604905,273.903708,21521800,2.970876,12.124905,276.685599,4.322386,285.330370,...,False,36.148376,273.854229,273.641086,0.213143,1.197686,-0.984543,4,12,4
2025-12-29,273.504089,274.103504,272.095404,272.435082,23715200,2.837061,12.102227,276.431337,4.353959,285.139255,...,False,39.068390,273.800361,273.630938,0.169423,0.992033,-0.822610,0,12,4


In [ ]:
# Add the seasonality features
stocks['DayOfWeek'] = stocks.index.dayofweek
stocks['Month'] = stocks.index.month
stocks['Quarter'] = stocks.index.quarter

#stocks.head()

In [ ]:
# Use the FED data to get the following features:
# - Federal Funds Rate
# - Inflation Rate
# - Unemployment Rate
%pip install fredapi
from fredapi import Fred
fred = Fred(api_key='c06a7676bffb4c8338823496ee6287bf')

interest_rate = fred.get_series('FEDFUNDS')
inflation_rate = fred.get_series('CPIAUCSL')
unemployment_rate = fred.get_series('UNRATE')

stocks['FederalFundsRate'] = interest_rate
stocks['InflationRate'] = inflation_rate
stocks['UnemploymentRate'] = unemployment_rate

stocks.head(20)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Price,Close,High,Low,Open,Volume,Gold,Bitcoin,MA20,STD20,UpperBand,...,EMA26,MACD,SignalLine,MACD_Histogram,DayOfWeek,Month,Quarter,FederalFundsRate,InflationRate,UnemploymentRate
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,72.400513,72.460776,71.156674,71.409778,135480400,1.000000,0.970181,NaN,NaN,NaN,...,72.400513,0.000000,0.000000,0.000000,3,1,1,NaN,NaN,NaN
2020-01-03,71.696648,72.455966,71.472469,71.629153,146322800,1.016202,1.020098,NaN,NaN,NaN,...,72.035044,-0.015792,-0.008773,-0.007019,4,1,1,NaN,NaN,NaN
2020-01-06,72.267921,72.306491,70.568495,70.819193,118387200,1.027353,1.079032,NaN,NaN,NaN,...,72.118715,-0.002407,-0.006164,0.003757,0,1,1,NaN,NaN,NaN
2020-01-07,71.928040,72.533080,71.708680,72.277563,108872000,1.031027,1.133819,NaN,NaN,NaN,...,72.065410,-0.008531,-0.006966,-0.001565,1,1,1,NaN,NaN,NaN
2020-01-08,73.085121,73.386438,71.631567,71.631567,132079200,1.021581,1.122176,NaN,NaN,NaN,...,72.301885,0.034363,0.005329,0.029035,2,1,1,NaN,NaN,NaN
2020-01-09,74.637474,74.830314,73.810661,74.061352,170108400,1.017842,1.094289,NaN,NaN,NaN,...,72.769685,0.125882,0.038005,0.087877,3,1,1,NaN,NaN,NaN
2020-01-10,74.806244,75.370316,74.304855,74.871333,140644800,1.021646,1.134216,NaN,NaN,NaN,...,73.131877,0.190052,0.076484,0.113567,4,1,1,NaN,NaN,NaN
2020-01-13,76.404427,76.430946,75.003904,75.122026,121532000,1.015677,1.131111,NaN,NaN,NaN,...,73.659166,0.306033,0.131649,0.174384,0,1,1,NaN,NaN,NaN
2020-01-14,75.372711,76.551468,75.249779,76.341752,161954400,1.011742,1.226049,NaN,NaN,NaN,...,73.913151,0.330504,0.177585,0.152918,1,1,1,NaN,NaN,NaN
